# T12 EDA: Kaggle 증상→질환 데이터

MediPath 베이스라인 모델(T14)에 쓸 Kaggle 대체 데이터의 특성을 확인합니다. 이 노트북의 숫자가 **평가 방식 변경(SPEC 1절)의 근거**입니다.

- 데이터: `data/raw/kaggle_disease_prediction/Training.csv`, `Testing.csv` (`python data/scripts/download.py`로 받음, 커밋하지 않음)
- 매핑: `data/mappings/` (T11)
- 그래프: `docs/figures/` (발표용, 한글 폰트 없이 그려지도록 라벨은 영어·ID)
- 실행: `pip install -r data/requirements-notebook.txt` 후 저장소 루트에서
  `python data/scripts/run_notebook.py data/notebooks/eda.ipynb`

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "SPEC.md").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "kaggle_disease_prediction"
MAP = ROOT / "data" / "mappings"
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "data" / "scripts"))
from check_mappings import symptom_id_from_column  # T11 정규화 규칙 재사용

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight"})


def load(name: str) -> pd.DataFrame:
    df = pd.read_csv(RAW / name)
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]]
    labels = df.pop("prognosis").str.strip()
    # 열 이름 정규화. 중복 열(fluid_overload)은 OR로 합칩니다.
    df.columns = [symptom_id_from_column(c) for c in df.columns]
    df = df.T.groupby(level=0).max().T.astype(int)
    df["prognosis"] = labels
    return df


train, test = load("Training.csv"), load("Testing.csv")
diseases = pd.read_csv(MAP / "disease_department.csv", dtype=str)
label_to_id = dict(zip(diseases["kaggle_label"], diseases["disease_id"]))
train["disease_id"], test["disease_id"] = (
    train.prognosis.map(label_to_id),
    test.prognosis.map(label_to_id),
)
SYMPTOMS = [c for c in train.columns if c not in ("prognosis", "disease_id")]
print("Training", train.shape, "| Testing", test.shape, "| symptoms", len(SYMPTOMS))
assert train.disease_id.notna().all() and test.disease_id.notna().all(), "unmapped label"

Training (4920, 133) | Testing (42, 133) | symptoms 131


## 1. 클래스 분포

![rows per disease](../../docs/figures/01_rows_per_disease.png)

In [2]:
counts = train.disease_id.value_counts().sort_index()
print(counts.describe()[["count", "min", "max"]])
ax = counts.plot.bar(figsize=(12, 3.2), color="#4C78A8")
ax.set(title="Rows per disease (Training.csv)", xlabel="", ylabel="rows")
plt.savefig(FIG / "01_rows_per_disease.png")
plt.close()

count     41.0
min      120.0
max      120.0
Name: count, dtype: float64


## 2. 중복과 누수 (평가 방식 변경의 근거)

![unique combos](../../docs/figures/02_unique_combos_per_disease.png)

같은 증상 조합이 여러 번 반복되면, 행 단위로 무작위 분할했을 때 같은 조합이 학습과 테스트에 동시에 들어갑니다.

In [3]:
key = train[SYMPTOMS].astype(str).agg("".join, axis=1)
test_key = test[SYMPTOMS].astype(str).agg("".join, axis=1)
unique_combos = key.nunique()
leak = test_key.isin(set(key)).sum()
ambiguous = train.assign(k=key).groupby("k").disease_id.nunique().gt(1).sum()
print(f"Training rows: {len(train)} / unique symptom combinations: {unique_combos}")
print(f"Testing rows also in Training (same combination): {leak} / {len(test)}")
print(f"Combinations mapped to more than one disease: {ambiguous}")
per_disease = train.assign(k=key).groupby("disease_id").k.nunique().sort_values()
print(per_disease.describe()[["min", "50%", "max"]])
ax = per_disease.plot.bar(figsize=(12, 3.2), color="#F58518")
ax.set(
    title=f"Unique symptom combinations per disease (total {unique_combos} of {len(train)} rows)",
    xlabel="",
    ylabel="unique combos",
)
plt.savefig(FIG / "02_unique_combos_per_disease.png")
plt.close()

Training rows: 4920 / unique symptom combinations: 304
Testing rows also in Training (same combination): 41 / 42
Combinations mapped to more than one disease: 0
min     5.0
50%     7.0
max    10.0
Name: k, dtype: float64


## 3. 증상 빈도 (고유 조합 기준)

![symptom frequency](../../docs/figures/03_symptom_frequency_top20.png)

In [4]:
dedup = train.drop_duplicates(subset=SYMPTOMS + ["disease_id"])
freq = dedup[SYMPTOMS].sum().sort_values(ascending=False)
print(
    "symptoms per combination:",
    dedup[SYMPTOMS].sum(axis=1).describe()[["min", "mean", "max"]].round(2).to_dict(),
)
print("symptoms never present:", list(freq[freq == 0].index))
ax = freq.head(20)[::-1].plot.barh(figsize=(6, 6), color="#54A24B")
ax.set(title="Top 20 symptoms (deduplicated combinations)", xlabel="combinations", ylabel="")
plt.savefig(FIG / "03_symptom_frequency_top20.png")
plt.close()

symptoms per combination: {'min': 3.0, 'mean': 7.65, 'max': 17.0}
symptoms never present: []


## 4. 증상 동시출현

![co-occurrence](../../docs/figures/04_cooccurrence_top20.png)

In [5]:
top = list(freq.head(20).index)
co = dedup[top].T.dot(dedup[top])
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(co.values, cmap="Blues")
ax.set_xticks(range(len(top)), top, rotation=90)
ax.set_yticks(range(len(top)), top)
ax.set_title("Co-occurrence of top 20 symptoms (deduplicated combinations)")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.savefig(FIG / "04_cooccurrence_top20.png")
plt.close()

## 5. 진료과 분포 (T11 매핑, 1순위 진료과)

![departments](../../docs/figures/05_department_distribution.png)

In [6]:
EN = {
    "01": "Internal Med",
    "02": "Neurology",
    "04": "Surgery",
    "05": "Orthopedics",
    "13": "ENT",
    "14": "Dermatology",
    "15": "Urology",
    "24": "Emergency Med",
}
dept = diseases.department_code.value_counts()
dept.index = [f"{c} {EN.get(c, c)}" for c in dept.index]
print(dept)
ax = dept[::-1].plot.barh(figsize=(6, 3.2), color="#B279A2")
ax.set(title="Diseases per primary department (draft mapping)", xlabel="diseases", ylabel="")
plt.savefig(FIG / "05_department_distribution.png")
plt.close()

01 Internal Med     24
14 Dermatology       6
05 Orthopedics       3
13 ENT               2
24 Emergency Med     2
04 Surgery           2
02 Neurology         1
15 Urology           1
Name: count, dtype: int64


## 6. 정리와 T14에 주는 의미
- **클래스 균형**: 41종 모두 120행이라 불균형 보정이 필요 없습니다.
- **중복**: 4,920행이 304개 고유 조합의 반복입니다. 행 단위 무작위 분할을 쓰면 안 되고, **고유 조합 단위 그룹 분할**이 필요합니다.
- **누수**: `Testing.csv` 42행 중 41행이 Training과 같은 조합이라 평가에 쓰지 않습니다.
- **조합이 적은 질환**: 질환당 고유 조합이 수 개뿐이라 그룹 분할 시 테스트셋이 작습니다. 그래서 **노이즈 테스트셋**(증상 제거·추가)으로 일반화를 함께 봅니다.
- **진료과 편중**: 41종 중 다수가 내과(01)라, 진료과 추천 정확도는 질환 Top-3 정확도보다 높게 나오기 쉽습니다. 해석할 때 주의합니다.